# 05 — MLP (PyTorch): escada de evolução E0 → E4
Experimentos do **Perceptron Multicamadas** nas 3 partições. A lógica está em `src/mlp.py`; este notebook só **chama** o módulo, registra os resultados em `reports/tables/resultados_mlp.csv` e mostra a evolução.

## Regras (para não invalidar o experimento)
- **Escolhas (hiperparâmetros, variante, etapa) usam somente o KS de validação** (desempate: AUROC de validação). O **teste só é registrado** ao lado, nunca usado para escolher.
- Busca de hiperparâmetros **apenas no Fold 1**; a melhor configuração é retreinada nos Folds 2 e 3 e reportamos **média ± desvio** entre folds.
- Sementes: busca com 1 semente fixa; as 5 melhores tentativas são reavaliadas com 3 sementes; resultado final = 3 folds × 3 sementes (média das sementes dentro de cada fold; o desvio entre sementes é mostrado à parte).
- Limiar fixo 0,5 para precisão/recall/F1 (KS e AUROC não dependem dele).
- **Perda é hiperparâmetro**: `cross_entropy` e `mse` entram na busca. Parada antecipada na perda de validação (paciência 20, máx. 10.000 épocas; melhora mínima 1e-4).

## Escada de evolução (registro do desempenho)
| Etapa | O que muda | Variante |
|---|---|---|
| **E0_baseline** | Rede pequena (1 camada, 10 unidades), **todas as variáveis** | `neural` |
| **E1_faixas** | Mesma rede + variáveis agregadas (faixas de `tenure` e `MonthlyCharges`) | `neural_faixas` |
| **E2_busca_neural** | Busca Optuna (100 tentativas) | `neural` |
| **E3_busca_faixas** | Busca Optuna (100 tentativas) | `neural_faixas` |
| **E4_final_3folds** | Melhor entre E2/E3 (por KS de validação), retreinada em 3 folds × 3 sementes | a escolhida |

## Tempo e retomada
A busca roda em CPU; espere algo entre **15 e 40 min por variante** (tentativas lentas, como SGD com taxa baixa, podem levar mais de 1 min). O estudo Optuna fica em `models/optuna/` (fora do Git): se interromper, **rodar a célula de novo continua de onde parou**. Os resultados ficam em `reports/tables/` e **vão para o Git**. No Colab, `models/` é apagado ao fim da sessão, mas os CSV/JSON em `reports/` bastam para reproduzir os números.

In [ ]:
# Localiza a raiz do repositório (funciona no VS Code e no Colab) e a coloca no sys.path
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src import config as cfg
from src import mlp
from src.metrics import (load_results, summarize_folds, evolution_table, plot_evolution,
                         plot_ks, plot_roc, plot_confusion, save_figure, save_results)

cfg.ensure_dirs()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---- Parâmetros do experimento (alterar apenas aqui) ----
FOLD_BUSCA = 1                    # a busca de hiperparâmetros é SOMENTE no Fold 1
N_TRIALS = 100                    # tentativas Optuna por variante (total; retoma se já existirem)
SEEDS = (42, 43, 44)              # 3 sementes para registro, reranqueamento e resultado final
TOP_K = 5                         # nº de tentativas reavaliadas com várias sementes
TIMEOUT_BUSCA = None              # segundos por variante (None = sem limite). Ex.: 1800 para 30 min
DEVICE = "cpu"                    # rede pequena: CPU é suficiente e determinística

print("PyTorch:", torch.__version__, "| dispositivo:", DEVICE, "| semente base:", cfg.SEED)

## E0 — Baseline: rede pequena, todas as variáveis
Primeiro modelo: **nenhuma variável é eliminada** (só o identificador `id_origem`, que nunca entrou nas features). Rede de 1 camada com 10 unidades, ReLU, Adam, entropia cruzada.

In [ ]:
X_tr, y_tr = mlp.load_xy(FOLD_BUSCA, "neural")["treino"]
# Confere que TODAS as 19 variáveis originais continuam representadas (numéricas ou one-hot)
faltando = [c for c in cfg.FEATURE_COLS if not any(col == c or col.startswith(c + "_") for col in X_tr.columns)]
assert not faltando, f"variáveis ausentes: {faltando}"
print(f"{len(cfg.FEATURE_COLS)} variáveis originais -> {X_tr.shape[1]} colunas após codificação")
print("Configuração E0:", mlp.config_to_dict(mlp.BASELINE))

res_e0, saidas_e0 = mlp.avaliar_configuracao(mlp.BASELINE, "neural", FOLD_BUSCA, SEEDS, etapa="E0_baseline",
                                             obs="rede pequena, todas as variáveis")
save_results(res_e0, cfg.REPORTS_TABLES)
res_e0[["etapa", "variante", "semente", "conjunto", "ks", "auroc", "mse", "log_loss", "f1", "recall", "precisao"]].round(4)

In [ ]:
# Curva de aprendizagem da primeira semente: onde parou e se há sobreajuste
r0 = saidas_e0[SEEDS[0]]["fit"]
print(f"melhor época: {r0.best_epoch} | épocas rodadas: {r0.n_epocas} | parou por paciência: {r0.parou_cedo}")
fig = mlp.plot_history(r0.history, r0.best_epoch, "E0 — curva de aprendizagem (Fold 1)")
save_figure(fig, "mlp_e0_curva_fold1")
plt.show()

## E1 — Variáveis agregadas (`neural_faixas`)
Mesma rede e mesmos hiperparâmetros do E0; **só muda a variante** (faixas de `tenure` e quartis de `MonthlyCharges`, ajustados no treino). Isolar essa mudança mostra quanto as variáveis agregadas ajudam.

In [ ]:
res_e1, _ = mlp.avaliar_configuracao(mlp.BASELINE, "neural_faixas", FOLD_BUSCA, SEEDS, etapa="E1_faixas",
                                     obs="mesma rede do E0 com faixas de tenure/MonthlyCharges")
save_results(res_e1, cfg.REPORTS_TABLES)
res_e1[["etapa", "variante", "semente", "conjunto", "ks", "auroc", "mse", "log_loss", "f1"]].round(4)

## E2 — Busca de hiperparâmetros na variante `neural`
Espaço de busca: 1–2 camadas, neurônios, ativação (logística/tanh/ReLU), otimizador (Adam/RMSprop/SGD/Adadelta) e taxa de aprendizagem, dropout, regularização L2, tamanho do lote e **perda (entropia cruzada vs MSE)**.
Objetivo: **maximizar o KS de validação**. Depois, as `TOP_K` melhores são reavaliadas com 3 sementes e a de maior KS médio é a escolhida (a tentativa isolada pode ser sorte de semente).

In [ ]:
out_e2 = mlp.etapa_busca("neural", "E2_busca_neural", n_trials=N_TRIALS, fold=FOLD_BUSCA, k=TOP_K,
                         seeds_rerank=SEEDS, seeds_registro=SEEDS, device=DEVICE, timeout=TIMEOUT_BUSCA)
print("Tentativas concluídas:", len(mlp.trials_table(out_e2["study"])))
print("\nTop tentativas reavaliadas com 3 sementes (só validação):")
display(out_e2["ranking"].round(4))
print("Configuração escolhida:", mlp.config_to_dict(out_e2["config"]))

In [ ]:
# Qual perda e qual otimizador funcionaram melhor entre as tentativas? (KS de validação)
t2 = mlp.trials_table(out_e2["study"])
display(t2.groupby("perda")["val_ks"].agg(["count", "mean", "max"]).round(4))
display(t2.groupby("otimizador")["val_ks"].agg(["count", "mean", "max"]).round(4))

## E3 — Busca de hiperparâmetros na variante `neural_faixas`
Mesmo procedimento, com busca própria (cada variante tem seus melhores hiperparâmetros).

In [ ]:
out_e3 = mlp.etapa_busca("neural_faixas", "E3_busca_faixas", n_trials=N_TRIALS, fold=FOLD_BUSCA, k=TOP_K,
                         seeds_rerank=SEEDS, seeds_registro=SEEDS, device=DEVICE, timeout=TIMEOUT_BUSCA)
print("Tentativas concluídas:", len(mlp.trials_table(out_e3["study"])))
display(out_e3["ranking"].round(4))
print("Configuração escolhida:", mlp.config_to_dict(out_e3["config"]))

In [ ]:
t3 = mlp.trials_table(out_e3["study"])
display(t3.groupby("perda")["val_ks"].agg(["count", "mean", "max"]).round(4))
display(t3.groupby("otimizador")["val_ks"].agg(["count", "mean", "max"]).round(4))

## E4 — Resultado final nos 3 folds
Escolhemos entre E2 e E3 **pelo KS médio de validação no Fold 1** (nunca pelo teste) e retreinamos a configuração vencedora nos Folds 1, 2 e 3 com 3 sementes cada.

In [ ]:
resultados = load_results()
etapa_vencedora, variante_final, comparacao = mlp.melhor_etapa(resultados, ["E2_busca_neural", "E3_busca_faixas"],
                                                               fold=FOLD_BUSCA)
print("Comparação (KS e AUROC de VALIDAÇÃO, Fold 1, média das sementes):")
display(comparacao.round(4))
print(f"Vencedora: {etapa_vencedora} (variante {variante_final})")

config_final = mlp.load_config(cfg.REPORTS_TABLES / f"mlp_melhor_{variante_final}.json")
res_e4, saidas_e4 = mlp.etapa_final(config_final, variante_final, etapa="E4_final_3folds", folds=(1, 2, 3), seeds=SEEDS)
print("Configuração final:", mlp.config_to_dict(config_final))

In [ ]:
resultados = load_results()
resultados_mlp = resultados[resultados["modelo"] == "mlp"]

# Média ± desvio ENTRE FOLDS (a média das sementes é feita dentro de cada fold); desvio entre sementes à parte
for conj in ("validacao", "teste"):
    s = summarize_folds(resultados_mlp[resultados_mlp["etapa"] == "E4_final_3folds"], conj)
    print(f"\n=== E4 — {conj} ===")
    display(s.round(4))

In [ ]:
# Resultado por fold (média das sementes) no teste
por_fold = (resultados_mlp[(resultados_mlp["etapa"] == "E4_final_3folds") & (resultados_mlp["conjunto"] == "teste")]
            .groupby("fold")[["ks", "auroc", "f1", "recall", "precisao", "mse", "log_loss"]].mean())
display(por_fold.round(4))

In [ ]:
# Gráficos do teste (Fold 1, 1ª semente): KS, ROC e matriz de confusão proporcional por linha
y_te, p_te = saidas_e4[1][SEEDS[0]]["scores"]["teste"]
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
plot_ks(y_te, p_te, "MLP final — teste, Fold 1", ax=axs[0])
plot_roc(y_te, p_te, "ROC — teste, Fold 1", ax=axs[1])
plot_confusion(y_te, p_te, title="Matriz de confusão — teste, Fold 1", ax=axs[2])
save_figure(fig, "mlp_final_teste_fold1")
plt.show()

## Evolução do desempenho
KS por etapa no **Fold 1**, nos conjuntos de validação e teste (barras = desvio entre sementes). **A escolha das configurações usou só a validação**; o teste aparece apenas como registro.

In [ ]:
resultados = load_results()
fig = plot_evolution(resultados, "mlp", metric="ks", fold=FOLD_BUSCA)
save_figure(fig, "mlp_evolucao_ks_fold1")
plt.show()
tabela_evolucao = evolution_table(resultados, "mlp", fold=FOLD_BUSCA, metrics=("ks", "auroc", "mse", "log_loss", "f1"))
tabela_evolucao.to_csv(cfg.REPORTS_TABLES / "mlp_evolucao_fold1.csv", index=False)
tabela_evolucao.round(4)

In [ ]:
# Mesma evolução para AUROC e MSE (MSE nos 3 conjuntos)
fig, axs = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
plot_evolution(resultados, "mlp", metric="auroc", fold=FOLD_BUSCA, conjuntos=("treino", "validacao", "teste"), ax=axs[0])
plot_evolution(resultados, "mlp", metric="mse", fold=FOLD_BUSCA, conjuntos=("treino", "validacao", "teste"), ax=axs[1])
save_figure(fig, "mlp_evolucao_auroc_mse_fold1")
plt.show()

## Registro do processo

| Etapa | O que mudou | KS validação | KS teste | Observação |
|---|---|---|---|---|
| E0_baseline | rede pequena (1 camada, 10 unidades, ReLU, Adam, entropia cruzada), todas as variáveis, `neural` | 0,504 | 0,514 | Ponto de partida. Parou na época ~20; o KS de validação estabiliza por volta da época 4. |
| E1_faixas | mesma rede + faixas de `tenure` e `MonthlyCharges` (`neural_faixas`) | 0,513 | 0,529 | Único ganho consistente (+0,01): as 3 sementes superaram o baseline, na validação e no teste. |
| E2_busca_neural | busca Optuna (100 tentativas) em `neural`; escolhida: 1 camada de 23 unidades, tanh, RMSprop, MSE | 0,515 | 0,518 | Dentro do ruído entre sementes (0,004 a 0,007). Sem ganho claro sobre o E1. |
| E3_busca_faixas | busca Optuna (100 tentativas) em `neural_faixas`; escolhida: 17-6 unidades, tanh, dropout 0,3, RMSprop, MSE | 0,522 | 0,530 | Teste igual ao do E1 (0,529): a busca não trouxe ganho real. Venceu o E2 pelo KS de validação. |
| E4_final_3folds | E3 retreinada em 3 folds × 3 sementes | 0,532 ± 0,015 | 0,540 ± 0,014 | Média ± desvio entre folds. AUROC de teste 0,843 ± 0,008. No Fold 1 é idêntico ao E3. |

**Conclusões**
- **Perda:** MSE superou entropia cruzada nas duas buscas (melhor KS de validação 0,518 contra 0,511 em `neural`; 0,527 contra 0,518 em `neural_faixas`). A diferença é pequena e o otimizador sorteou mais tentativas com MSE, então a evidência é moderada.
- **Otimizador e ativação:** RMSprop com tanh dominou as 30 melhores tentativas.
- **Tamanho da rede:** redes maiores não ajudaram. Os melhores modelos têm 1 ou 2 camadas pequenas e a melhor época fica entre 1 e 8, ou seja, a rede sobreajusta rápido.
- **Variáveis agregadas:** melhoraram o KS (~+0,01), e foi o único ganho consistente.
- **Treino versus validação:** o KS de treino fica ~0,05 acima do de validação em todas as etapas, um sobreajuste moderado e estável.
- **Teto:** as 10 melhores tentativas de cada busca diferem só 0,003 a 0,004 de KS. Mais busca não deve ajudar; ganhos futuros virão de engenharia de variáveis.
- **Limitação:** MSE e log-loss do teste não são comparáveis aos da validação, porque o modelo treina com 50% de churn e o teste tem 26,5%. KS e AUROC não sofrem com isso.

## Notas de reprodutibilidade
- Dados: `data/processed/<variante>/` (mesmos folds para todos os modelos). Semente base 42 (`src/config.py`).
- Resultados versionados: `reports/tables/resultados_mlp.csv`, `optuna_mlp_<variante>_fold1_trials.csv`, `mlp_melhor_<variante>.json`, `mlp_evolucao_fold1.csv`.
- Rodar a mesma célula de novo **substitui** as linhas com a mesma chave (modelo, etapa, variante, fold, semente, conjunto): não duplica.